# acceleration and jerk
"Fit" or estimate a polynomial of time to explain the residuals away from the clock fourier fit; interpret as an acceleration (and maybe in future jerk, snap, crackle, and pop).

## bugs and to-dos:
- This code should not run best_clocks_in_star(); this should read from the clocks database.
- Only does acceleration, no jerk.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt

In [2]:
# all this to import the KeplerClocks code
from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks

/Users/dhogg/.pyenv/versions/3.13.12/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [3]:
kicid = "KIC009899416"
clock_table = clocks.best_clocks_in_star(kicid)
print(clock_table)

clocks.best_clocks_in_star(): getting Kepler data for KIC009899416
clocks.get_kepler_data(): starting to download data for KIC009899416
clocks.get_kepler_data() took 11.517422914505005 s
clocks.best_clocks_in_star(): getting candidate frequencies for KIC009899416
clocks.best_clocks_in_star(): getting clock values for KIC009899416
clocks.best_clocks_in_star(): building long clock table for KIC009899416
clocks.best_clocks_in_star(): trimming clock table for KIC009899416
identify_resonances(): 1.5717003740145072 2.357549989701148 1.4999996364952108 3/2 0.6666668282243898 2/3 True
identify_resonances(): 1.5717003740145072 4.715124793514926 3.0000150610585807 3 0.3333316598907812 1/3 True
identify_resonances(): 1.5717003740145072 0.9430054583367077 0.5999906050337324 3/5 1.6666927642038303 5/3 True
identify_resonances(): 1.5717003740145072 3.1434063850372267 2.0000035865666925 2 0.49999910335993475 1/2 True
identify_resonances(): 1.5717003740145072 9.430214915316256 6.0000080621150325 6 0.1

In [27]:
def acceleration(row, njack=8):
    # unpack
    kicid, om, M, value = row['kicid'], row['angular_frequency'], row['fourier_series_degree'], row['theoretical_value']

    # get data
    ts, ys, errs, deltaf, deltat = clocks.get_kepler_data(kicid)
    ivars = 1. / errs ** 2
    X, ms, pars = clocks.fourier_wls_fit(om, M, ts, ys, ivars)
    resids = ys - X @ pars
    deriv_pars = clocks.take_derivative_wrt_phase(pars, ms)
    derivs = om * X @ deriv_pars # dflux / dt

    # make design matrix for acceleration model
    t0 = np.sum(ivars * ts) / np.sum(ivars)  # ivar-weighted mean time
    A = np.vstack((1. * derivs, (ts - t0) * derivs, 0.5 * (ts - t0) ** 2 * derivs)).T
    _, d = A.shape
    
    # start jackknife
    subsamp = np.arange(len(ys)) * njack // len(ys)
    pars = np.zeros((njack, d)) + np.nan
    for samp in range(njack):
        loo = (subsamp != samp)
        Aloo = A[loo, :]
        ATA = Aloo.T @ (ivars[loo, None] * Aloo)
        ATY = Aloo.T @ (ivars[loo] * ys[loo])
        pars[samp] = np.linalg.solve(ATA, ATY)
    meanpars = np.mean(pars, axis=0)
    diagvars = ((njack - 1) / njack) * np.sum((pars - meanpars[None, :]) ** 2, axis=0)
    return meanpars, diagvars

In [28]:
for row in clock_table:
    pars, pars_vars = acceleration(row)
    print(pars, np.sqrt(pars_vars))

clocks.get_kepler_data(): starting to download data for KIC009899416
clocks.get_kepler_data() took 0.6273288726806641 s
[-9.62569739e-05  4.85019912e-07  5.99956783e-10] [2.03905488e-04 3.99873297e-07 1.97520729e-09]
